# Lesson 15 – Distillation for Weird AI

Chapter 8 introduces **model distillation**. Distillation trains a smaller student model using outputs from a stronger teacher model.

## 1. What Is Distillation?

A teacher model produces examples. A student model learns from those examples.

```text
Teacher model → Synthetic examples → Student model training
```

In [ ]:
teacher_output = {
    "prompt": "Write an emo parody about database indexes.",
    "teacher_notes": "Use database vocabulary, emotional tone, and paired rhymes.",
    "teacher_lyrics": "My index broke tonight\nThe query lost the fight\nRows vanished out of sight\nI cried in candlelight",
    "quality_score": 0.92,
}
teacher_output

## 2. Hard vs. Soft Distillation

**Hard distillation** trains on the teacher's generated text.

**Soft distillation** trains on the teacher's probability distribution over possible next tokens.

In [ ]:
hard_distillation_target = teacher_output["teacher_lyrics"]
soft_distillation_target = {"night": 0.55, "fight": 0.25, "light": 0.15, "query": 0.05}
print("Hard target:")
print(hard_distillation_target)
print("\nSoft target distribution:")
print(soft_distillation_target)

## 3. Distillation vs. Reinforcement Learning

RL asks: *Did the output score well?*

Distillation asks: *Can the student reproduce this high-quality example?*

## 4. Why Distillation Fits Weird AI

Weird AI can use generation, evaluation, best-of-N selection, and self-refinement to create high-quality examples. Distillation lets us save those examples and train a future model to imitate them.

In [ ]:
examples = [
    {"prompt": "Write about SQL joins", "quality_score": 0.91},
    {"prompt": "Write about recursion", "quality_score": 0.88},
    {"prompt": "Write about Docker", "quality_score": 0.72},
]
[ex for ex in examples if ex["quality_score"] >= 0.8]

## 5. Teacher Notes as `<think>` Traces

The book formats teacher reasoning traces using `<think>...</think>` tags. For Weird AI, these can hold teacher notes about rhyme, tone, and structure.

In [ ]:
def format_teacher_response(lyrics, notes, include_thinking=True):
    lyrics = lyrics.strip()
    notes = notes.strip()
    if not include_thinking:
        return lyrics
    return f"""<think>\n{notes}\n</think>\n\n{lyrics}""".strip()

formatted = format_teacher_response(teacher_output["teacher_lyrics"], teacher_output["teacher_notes"])
print(formatted)

## 6. Building a Full Training Example

A distillation training example combines the prompt and the teacher response.

In [ ]:
def format_training_prompt(prompt):
    return f"User: {prompt.strip()}\nAssistant:"

full_text = format_training_prompt(teacher_output["prompt"]) + "\n" + formatted
print(full_text)

## 7. Why Mask Prompt Tokens?

During training, we usually want the model to learn the teacher response, not memorize the prompt. Prompt labels are often set to `-100` so the loss ignores them.

In [ ]:
prompt_tokens = [10, 11, 12]
target_tokens = [20, 21, 22, 23]
input_ids = prompt_tokens + target_tokens
labels = [-100] * len(prompt_tokens) + target_tokens
print("input_ids:", input_ids)
print("labels:   ", labels)

## 8. JSONL Dataset Format

JSONL stores one JSON object per line. It is convenient for training datasets because files can be streamed line by line.

In [ ]:
import json
record = {
    "prompt": teacher_output["prompt"],
    "target_text": formatted,
    "input_ids": input_ids,
    "labels": labels,
    "prompt_length": len(prompt_tokens),
    "metadata": {"quality_score": teacher_output["quality_score"]},
}
print(json.dumps(record, indent=2))

## 9. Course Wrap-Up

Weird AI has evolved from a lyric generator into an AI engineering pipeline: generation, evaluation, best-of-N, self-refinement, rewards, diagnostics, and distillation dataset preparation.

## Reflection

1. What is model distillation?
2. Why is hard distillation easier to use than soft distillation?
3. How is distillation different from reinforcement learning?
4. Why might distillation be more practical for Weird AI than reinforcement learning?
5. What risks come from training on teacher-generated examples?